# Phase 2: Virtual Library Expansion & Lipinski Filtering

**Objective:** Generate a combinatorial library of taloside derivatives using click chemistry rules, then apply drug-likeness filtering.

**Output:** 
- 500-1000 synthetic compounds (SMILES strings)
- Descriptors (MW, LogP, TPSA, H-donors, H-acceptors, rotatable bonds)
- Lipinski pass/fail classification
- Visual analysis of chemical space

---

## 1. Setup & Imports

In [ ]:
# Standard library
import sys
from pathlib import Path
import logging

# Data science
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cheminformatics
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, Crippen

# Visualization
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("[OK] All imports successful")

## 2. Import the GlycoLibraryGenerator

In [ ]:
# Add current directory to path if needed
if '.' not in sys.path:
    sys.path.insert(0, '.')

# Import from our pipeline
from glycolibrary_generator import (
    GlycoLibraryGenerator,
    LibraryConfig,
    ReactionSMARTS,
    configure_logging
)

print("[OK] GlycoLibraryGenerator imported")

## 3. Define Scaffold & Building Blocks

### Scaffold: Core Taloside Structure

This is the **parent molecule** from your MSc research:
- **Carbohydrate backbone**: Lactose-like core with methylation
- **Triazole connector**: Pre-installed click chemistry handle
- **Terminal alkyne**: Positioned for synthetic elaboration via click chemistry

### Building Blocks: Aromatic Alkynes

These are **functional group libraries** representing commercially available or easily synthesizable alkynes. We'll vary:
- **Electronic effects** (electron-donating OMe vs. electron-withdrawing Cl, F)
- **Regiochemistry** (ortho, meta, para positioning)
- **Heteroaromaticity** (pyridine, furan for bioisosteres)

In [ ]:
# ============================================================================
# CORE TALOSIDE SCAFFOLD
# ============================================================================

# This is the taloside scaffold with an azide group for click chemistry
# Structure breakdown:
#   - O=C(O[C@H]1...)  = Carbamate protecting group on the carbohydrate
#   - [C@@H](OCN=[N+]=[N-])  = Azide linker for click chemistry
#   - Gal-3 targeting payload

SCAFFOLD_SMILES = (
    "O=C(O[C@H]1[C@@H](OCN=[N+]=[N-])[C@@H](O)[C@@H](CO)O[C@H]1OC)C4=C([N+]([O-])=O)C=CC=C4"
)

print(f"Scaffold SMILES: {SCAFFOLD_SMILES}")
print(f"Scaffold length: {len(SCAFFOLD_SMILES)} characters")

# Validate scaffold
scaffold_mol = Chem.MolFromSmiles(SCAFFOLD_SMILES)
if scaffold_mol:
    print(f"[OK] Scaffold valid")
    print(f"  Atoms: {scaffold_mol.GetNumAtoms()}")
    print(f"  Heavy atoms: {scaffold_mol.GetNumHeavyAtoms()}")
    print(f"  Molecular weight: {Descriptors.MolWt(scaffold_mol):.2f} Da")
else:
    print(f"[X] Invalid scaffold SMILES")

In [ ]:
# ============================================================================
# BUILDING BLOCK LIBRARY: Aromatic Alkynes
# ============================================================================

# Strategy: Generate chemistry-driven building blocks, NOT random sampling
# Each alkyne represents a distinct chemical modification

BUILDING_BLOCKS = [
    # Neutral reference
    {'id': 'BB-001-Ph', 'smiles': 'C#Cc1ccccc1', 'description': 'Phenyl (neutral baseline)'},
    
    # Electron-donating substituents (increase nucleophilicity)
    {'id': 'BB-002-4OMe', 'smiles': 'C#Cc1ccc(OC)cc1', 'description': '4-Methoxy (EDG, para)'},
    {'id': 'BB-003-3OMe', 'smiles': 'C#Cc1cc(OC)ccc1', 'description': '3-Methoxy (EDG, meta)'},
    {'id': 'BB-004-2OMe', 'smiles': 'C#Cc1ccccc1OC', 'description': '2-Methoxy (EDG, ortho)'},
    
    # Electron-withdrawing substituents (decrease nucleophilicity)
    {'id': 'BB-005-4Cl', 'smiles': 'C#Cc1ccc(Cl)cc1', 'description': '4-Chloro (EWG, para)'},
    {'id': 'BB-006-4F', 'smiles': 'C#Cc1ccc(F)cc1', 'description': '4-Fluoro (EWG, para)'},
    {'id': 'BB-007-4Br', 'smiles': 'C#Cc1ccc(Br)cc1', 'description': '4-Bromo (EWG, para)'},
    {'id': 'BB-008-3Cl', 'smiles': 'C#Cc1cc(Cl)ccc1', 'description': '3-Chloro (EWG, meta)'},
    {'id': 'BB-009-3Br', 'smiles': 'C#Cc1cc(Br)ccc1', 'description': '3-Bromo (EWG, meta)'},
    {'id': 'BB-010-2Cl', 'smiles': 'C#Cc1ccccc1Cl', 'description': '2-Chloro (EWG, ortho - steric)'},
    
    # Heteroaromatic bioisosteres
    {'id': 'BB-011-2Pyr', 'smiles': 'C#Cc1ccccn1', 'description': '2-Pyridyl (heterocycle)'},
    {'id': 'BB-012-3Pyr', 'smiles': 'C#Cc1cccnc1', 'description': '3-Pyridyl (heterocycle)'},
    {'id': 'BB-013-4Pyr', 'smiles': 'C#Cc1ccncc1', 'description': '4-Pyridyl (heterocycle)'},
    {'id': 'BB-014-Furan', 'smiles': 'C#Cc1ccoc1', 'description': 'Furan-2-yl (heterocycle)'},
    {'id': 'BB-015-Thioph', 'smiles': 'C#Cc1ccsc1', 'description': 'Thiophen-2-yl (heterocycle)'},
    
    # Polysubstituted arenes
    {'id': 'BB-016-35diMe', 'smiles': 'C#Cc1cc(C)cc(C)c1', 'description': '3,5-Dimethyl (bulky EDG)'},
    {'id': 'BB-017-34diCl', 'smiles': 'C#Cc1cc(Cl)c(Cl)cc1', 'description': '3,4-Dichloro (strong EWG)'},
    {'id': 'BB-018-24diMe', 'smiles': 'C#Cc1ccccc1C', 'description': '2,4-Dimethyl (EDG, ortho+para)'},
    
    # Extended π-systems (absorption tuning)
    {'id': 'BB-019-Naphth', 'smiles': 'C#Cc1ccc2ccccc2c1', 'description': 'Naphthyl (extended conjugation)'},
    {'id': 'BB-020-Styryl', 'smiles': 'C#CC=Cc1ccccc1', 'description': 'Styryl (sp2 linker)'},
]

print(f"Building Block Library: {len(BUILDING_BLOCKS)} compounds\n")
print("Building Block Details:")
print("-" * 80)

for bb in BUILDING_BLOCKS:
    mol = Chem.MolFromSmiles(bb['smiles'])
    if mol:
        print(f"{bb['id']:15} | {bb['description']:30} | MW: {Descriptors.MolWt(mol):6.2f}")
    else:
        print(f"{bb['id']:15} | INVALID SMILES")

print("-" * 80)
print(f"Total building blocks: {len(BUILDING_BLOCKS)}")

## 4. Generate the Virtual Library

**What happens here:**

1. For each building block (20 total)
2. Apply the click chemistry reaction (triazole formation)
3. Check: Is the product chemically valid? (no hypervalency, correct MW)
4. Calculate descriptors (LogP, TPSA, etc.)
5. Remove duplicates

**Expected output:** ~20 products (one per building block), with full descriptor panel

In [ ]:
# Configure library generation
config = LibraryConfig(
    max_products=500,           # Safety limit
    min_product_mw=200.0,       # Minimum reasonable molecule
    max_product_mw=1000.0,      # Taloside can be large
    include_stereoisomers=True, # Preserve chiral centers
    filter_hypervalent=True,    # Remove chemically invalid
    output_dir=Path("phase2_notebook_output")
)

print(f"Config: {config}")

In [ ]:
# Create logger for notebook (less verbose)
logger_nb = configure_logging(log_file=Path("phase2_notebook.log"))
logger_nb.setLevel(logging.INFO)

print("Logger configured")

In [ ]:
# Instantiate generator
try:
    generator = GlycoLibraryGenerator(
        scaffold_smiles=SCAFFOLD_SMILES,
        building_blocks=BUILDING_BLOCKS,
        reaction_smarts=ReactionSMARTS.TRIAZOLE_FORMATION["smarts"],
        config=config,
        logger_instance=logger_nb
    )
    print("[OK] Generator instantiated successfully")
except Exception as e:
    print(f"[X] Error: {e}")
    raise

In [ ]:
# Generate library (this takes a few seconds)
print("\n[GENERATING VIRTUAL LIBRARY...]\n")

library_df = generator.generate_library()

print(f"\n[OK] Generation complete!")
print(f"  Total products: {len(library_df)}")
print(f"  Failed products: {len(generator.failed_products)}")

In [ ]:
# Display the generated library
print("\n" + "="*100)
print("GENERATED LIBRARY SUMMARY")
print("="*100 + "\n")

print(library_df.head(10).to_string())

In [ ]:
# Show a few example product SMILES
print("\nExample Products (SMILES):")
print("-" * 100)

for idx, row in library_df.head(5).iterrows():
    print(f"\n{row['compound_id']}:")
    print(f"  SMILES: {row['product_smiles'][:80]}...")
    print(f"  MW: {row['molecular_weight']:.2f} | LogP: {row['logp']:.2f} | TPSA: {row['tpsa']:.2f}")
    print(f"  H-donors: {row['h_donors']} | H-acceptors: {row['h_acceptors']}")

## 5. Descriptive Statistics of Generated Library

In [ ]:
# Descriptor statistics
print("\nDESCRIPTOR STATISTICS\n")
print("="*80)

descriptor_cols = ['molecular_weight', 'logp', 'h_donors', 'h_acceptors', 'tpsa', 'rotatable_bonds']
stats_df = library_df[descriptor_cols].describe().round(2)

print(stats_df.to_string())

print("\n" + "="*80)

In [ ]:
# Distribution by building block
print("\nProducts per Building Block:")
print("-" * 50)

bb_counts = library_df['building_block_id'].value_counts().sort_index()
for bb_id, count in bb_counts.items():
    print(f"  {bb_id:15} → {count:3} product(s)")

## 6. Apply Lipinski's Rule of 5 Filtering

In [ ]:
def apply_lipinski_filter(df, strict_mode=True):
    """
    Apply Lipinski's Rule of 5 filters.
    
    Lipinski's Rules:
    - Molecular Weight ≤ 500 Da (≤600 for carbohydrates)
    - LogP ≤ 5
    - H-bond donors ≤ 5
    - H-bond acceptors ≤ 10
    
    Args:
        df: DataFrame with descriptor columns
        strict_mode: If True, use carbohydrate-adjusted thresholds
        
    Returns:
        Tuple (passed_df, failed_df)
    """
    
    if strict_mode:
        # Adjusted for carbohydrate drugs (more polar, higher MW tolerated)
        mw_threshold = 600
        logp_threshold = 4.0
        hbd_threshold = 6
        hba_threshold = 12
    else:
        # Classical Lipinski
        mw_threshold = 500
        logp_threshold = 5.0
        hbd_threshold = 5
        hba_threshold = 10
    
    # Create pass/fail columns
    df_copy = df.copy()
    df_copy['passes_mw'] = df_copy['molecular_weight'] <= mw_threshold
    df_copy['passes_logp'] = df_copy['logp'] <= logp_threshold
    df_copy['passes_hbd'] = df_copy['h_donors'] <= hbd_threshold
    df_copy['passes_hba'] = df_copy['h_acceptors'] <= hba_threshold
    
    df_copy['passes_all'] = (df_copy['passes_mw'] & df_copy['passes_logp'] & 
                             df_copy['passes_hbd'] & df_copy['passes_hba'])
    
    passed = df_copy[df_copy['passes_all']].copy()
    failed = df_copy[~df_copy['passes_all']].copy()
    
    return passed, failed, df_copy

# Apply filtering
lipinski_passed, lipinski_failed, df_with_flags = apply_lipinski_filter(
    library_df, 
    strict_mode=True  # Carbohydrate-adjusted thresholds
)

print("\nLIPINSKI FILTERING RESULTS\n")
print("="*80)
print(f"Total compounds:     {len(library_df)}")
print(f"Passed Lipinski:     {len(lipinski_passed)} ({100*len(lipinski_passed)/len(library_df):.1f}%)")
print(f"Failed Lipinski:     {len(lipinski_failed)} ({100*len(lipinski_failed)/len(library_df):.1f}%)")
print("="*80)

In [ ]:
# Show failure reasons
print("\nFAILURE MODE ANALYSIS\n")
print("-" * 80)

if len(lipinski_failed) > 0:
    print("\nCompounds Failing Lipinski:")
    print("-" * 80)
    
    for idx, row in lipinski_failed.iterrows():
        failures = []
        if not row['passes_mw']:
            failures.append(f"MW={row['molecular_weight']:.1f} (threshold: 600)")
        if not row['passes_logp']:
            failures.append(f"LogP={row['logp']:.2f} (threshold: 4.0)")
        if not row['passes_hbd']:
            failures.append(f"HBD={row['h_donors']} (threshold: 6)")
        if not row['passes_hba']:
            failures.append(f"HBA={row['h_acceptors']} (threshold: 12)")
        
        print(f"\n{row['compound_id']}")
        print(f"  Failed: {', '.join(failures)}")
else:
    print("[OK] All compounds pass Lipinski filtering!")

## 7. Visualization: Descriptor Distributions

In [ ]:
plt.suptitle('Descriptor Distributions: Generated Virtual Library', 
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('descriptor_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print("[OK] Saved: descriptor_distributions.png")

## 8. Visualization: Lipinski Pass/Fail Breakdown

In [ ]:
plt.suptitle('Lipinski Rule of 5 Compliance Breakdown', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('lipinski_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

print("[OK] Saved: lipinski_breakdown.png")

## 9. Building Block Contribution Analysis

In [ ]:
# Analyze descriptor distributions by building block type
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Categorize building blocks
def categorize_bb(bb_id):
    if 'OMe' in bb_id or 'diMe' in bb_id:
        return 'Electron-Donating'
    elif 'Cl' in bb_id or 'F' in bb_id or 'Br' in bb_id or 'diCl' in bb_id:
        return 'Electron-Withdrawing'
    elif 'Pyr' in bb_id or 'Furan' in bb_id or 'Thioph' in bb_id:
        return 'Heteroaromatic'
    elif 'Naphth' in bb_id or 'Styryl' in bb_id:
        return 'Extended π'
    else:
        return 'Neutral'

library_df['bb_category'] = library_df['building_block_id'].apply(categorize_bb)

# 1. MW by category
ax = axes[0]
bb_categories = library_df['bb_category'].unique()
colors_cat = plt.cm.Set3(np.linspace(0, 1, len(bb_categories)))

for i, cat in enumerate(sorted(bb_categories)):
    data = library_df[library_df['bb_category'] == cat]['molecular_weight']
    ax.scatter([i] * len(data), data, alpha=0.6, s=100, color=colors_cat[i], label=cat, edgecolors='black')

ax.axhline(600, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Lipinski threshold')
ax.set_xticks(range(len(sorted(bb_categories))))
ax.set_xticklabels(sorted(bb_categories), rotation=45, ha='right')
ax.set_ylabel('Molecular Weight (Da)', fontsize=11, fontweight='bold')
ax.set_title('MW by Building Block Type', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.legend(fontsize=8, loc='upper left')

# 2. LogP by category
ax = axes[1]
for i, cat in enumerate(sorted(bb_categories)):
    data = library_df[library_df['bb_category'] == cat]['logp']
    ax.scatter([i] * len(data), data, alpha=0.6, s=100, color=colors_cat[i], label=cat, edgecolors='black')

ax.axhline(4.0, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Lipinski threshold')
ax.set_xticks(range(len(sorted(bb_categories))))
ax.set_xticklabels(sorted(bb_categories), rotation=45, ha='right')
ax.set_ylabel('LogP', fontsize=11, fontweight='bold')
ax.set_title('LogP by Building Block Type', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.legend(fontsize=8, loc='upper left')

# 3. TPSA by category (polarity)
ax = axes[2]
for i, cat in enumerate(sorted(bb_categories)):
    data = library_df[library_df['bb_category'] == cat]['tpsa']
    ax.scatter([i] * len(data), data, alpha=0.6, s=100, color=colors_cat[i], label=cat, edgecolors='black')

ax.axhline(140, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Threshold')
ax.set_xticks(range(len(sorted(bb_categories))))
ax.set_xticklabels(sorted(bb_categories), rotation=45, ha='right')
ax.set_ylabel('TPSA (Ų)', fontsize=11, fontweight='bold')
ax.set_title('Polarity by Building Block Type', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.legend(fontsize=8, loc='upper left')

plt.suptitle('Building Block Impact on Product Properties', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('building_block_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: building_block_analysis.png")

In [ ]:
# Statistical summary by building block category
print("\nDESCRIPTOR STATISTICS BY BUILDING BLOCK TYPE\n")
print("="*100)

for cat in sorted(library_df['bb_category'].unique()):
    subset = library_df[library_df['bb_category'] == cat]
    print(f"\n{cat.upper()} (n={len(subset)})")
    print("-" * 100)
    print(subset[['molecular_weight', 'logp', 'h_donors', 'h_acceptors', 'tpsa', 'rotatable_bonds']].describe().round(2).to_string())

## 10. Comparison: Original vs. Generated Compounds

Let's compare the **original 10 taloside compounds** from your MSc work to the **virtually generated library**.

In [ ]:
# Original taloside compounds from your MSc work
original_compounds = {
    "Talo-1": "O=C(O[C@H]1[C@@H](OCN2C=C(C3=CC(OC)=CC=C3)N=N2)[C@@H](O)[C@@H](CO)O[C@H]1OC)C4=C([N+]([O-])=O)C=CC=C4",
    "Talo-2": "O=C(O[C@H]1[C@@H](OCN2C=C(C3=C(OC)C=CC=C3)N=N2)[C@@H](O)[C@@H](CO)O[C@H]1OC)C4=C([N+]([O-])=O)C=CC=C4",
    "Talo-3": "O=C(O[C@H]1[C@@H](OCN2C=C(C3=CC=CC=C3)N=N2)[C@@H](O)[C@@H](CO)O[C@H]1OC)C4=C([N+]([O-])=O)C=CC=C4",
    "Talo-4": "O=C(O[C@H]1[C@@H](OCN2C=C(C3=CC=C(OC)C=C3)N=N2)[C@@H](O)[C@@H](CO)O[C@H]1OC)C4=C([N+]([O-])=O)C=CC=C4",
    "Talo-5": "O=C(O[C@H]1[C@@H](OCN2C=C(C3=CN=CC=C3)N=N2)[C@@H](O)[C@@H](CO)O[C@H]1OC)C4=C([N+]([O-])=O)C=CC=C4",
}

# Calculate descriptors for original compounds
original_data = []
for name, smiles in original_compounds.items():
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        original_data.append({
            'compound_id': name,
            'type': 'Original (MSc)',
            'molecular_weight': Descriptors.MolWt(mol),
            'logp': Crippen.MolLogP(mol),
            'h_donors': Lipinski.NumHDonors(mol),
            'h_acceptors': Lipinski.NumHAcceptors(mol),
            'tpsa': Descriptors.TPSA(mol),
            'rotatable_bonds': Lipinski.NumRotatableBonds(mol),
        })

original_df = pd.DataFrame(original_data)

# Compare with generated (add type label)
generated_df_copy = library_df.copy()
generated_df_copy['type'] = 'Generated (Virtual)'

# Combine
comparison_df = pd.concat([
    original_df,
    generated_df_copy[['compound_id', 'type', 'molecular_weight', 'logp', 'h_donors', 'h_acceptors', 'tpsa', 'rotatable_bonds']]
], ignore_index=True)

print("\nCOMPARISON: Original vs. Generated Compounds\n")
print("="*100)
print(f"\nOriginal compounds (MSc): {len(original_df)}")
print(original_df[['compound_id', 'molecular_weight', 'logp', 'h_donors', 'h_acceptors', 'tpsa']].to_string(index=False))

print(f"\nGenerated compounds (Virtual): {len(generated_df_copy)}")
print(generated_df_copy[['compound_id', 'molecular_weight', 'logp', 'h_donors', 'h_acceptors', 'tpsa']].head(10).to_string(index=False))
print("...")

print("\n" + "="*100)
print("Statistical Comparison:")
print("="*100)

for desc in ['molecular_weight', 'logp', 'h_donors', 'h_acceptors', 'tpsa', 'rotatable_bonds']:
    orig_mean = original_df[desc].mean()
    gen_mean = generated_df_copy[desc].mean()
    
    print(f"\n{desc.upper()}:")
    print(f"  Original mean:  {orig_mean:.2f}")
    print(f"  Generated mean: {gen_mean:.2f}")
    print(f"  Difference:     {gen_mean - orig_mean:+.2f}")

In [ ]:
# Visualization: Compare original vs generated
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

descriptors_compare = [
    ('molecular_weight', 'Molecular Weight (Da)'),
    ('logp', 'LogP'),
    ('h_donors', 'H-Bond Donors'),
    ('h_acceptors', 'H-Bond Acceptors'),
    ('tpsa', 'TPSA (Ų)'),
    ('rotatable_bonds', 'Rotatable Bonds'),
]

for idx, (col, label) in enumerate(descriptors_compare):
    ax = axes[idx // 3, idx % 3]
    
    # Violin plot
    parts = ax.violinplot(
        [generated_df_copy[col], original_df[col]],
        positions=[1, 2],
        showmeans=True,
        showmedians=True
    )
    
    # Overlay individual points
    ax.scatter([1] * len(generated_df_copy), generated_df_copy[col], 
               alpha=0.5, s=50, color='steelblue', label='Generated', edgecolors='black')
    ax.scatter([2] * len(original_df), original_df[col], 
               alpha=0.8, s=100, color='orange', label='Original (MSc)', edgecolors='black', marker='^')
    
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Generated\n(n={})'.format(len(generated_df_copy)), 
                        'Original\n(n={})'.format(len(original_df))])
    ax.set_ylabel(label, fontsize=11, fontweight='bold')
    ax.set_title(f'{label}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    if idx == 0:
        ax.legend(fontsize=9, loc='upper right')

plt.suptitle('Descriptor Comparison: Original (MSc) vs. Generated (Virtual) Library', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('original_vs_generated_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: original_vs_generated_comparison.png")

## 11. Export Results to CSV Files

In [ ]:
# Export all results
config.output_dir.mkdir(parents=True, exist_ok=True)

# 1. All generated compounds
all_compounds_path = config.output_dir / "01_all_generated_compounds.csv"
library_df.to_csv(all_compounds_path, index=False)
print(f"[OK] Exported: {all_compounds_path}")
print(f"  {len(library_df)} compounds")

# 2. Lipinski passed
lipinski_passed_path = config.output_dir / "02_lipinski_passed.csv"
lipinski_passed.drop(columns=['passes_mw', 'passes_logp', 'passes_hbd', 'passes_hba', 'passes_all'], 
                     errors='ignore').to_csv(lipinski_passed_path, index=False)
print(f"\n[OK] Exported: {lipinski_passed_path}")
print(f"  {len(lipinski_passed)} compounds (drug-like)")

# 3. Lipinski failed
lipinski_failed_path = config.output_dir / "03_lipinski_failed.csv"
lipinski_failed.drop(columns=['passes_mw', 'passes_logp', 'passes_hbd', 'passes_hba', 'passes_all'], 
                     errors='ignore').to_csv(lipinski_failed_path, index=False)
print(f"\n[OK] Exported: {lipinski_failed_path}")
print(f"  {len(lipinski_failed)} compounds (non-drug-like)")

# 4. Failed products (from generator)
generator.export_failed_products()

print("\n" + "="*80)
print("All CSV files exported to: {}".format(config.output_dir.absolute()))
print("="*80)

## 12. Lead Compound Identification

From the generated library, identify **promising lead candidates** based on:
- Drug-likeness (Lipinski pass)
- Structural diversity
- Balanced descriptor profile

In [ ]:
# Identify lead compounds (Lipinski-passing + good descriptor balance)
def rank_lead_compounds(df, n_leads=10):
    """
    Rank compounds by drug-likeness and descriptor balance.
    
    Scoring criteria:
    - MW: closer to 400-500 is better
    - LogP: closer to 2-3 is better (good balance)
    - TPSA: 60-120 is ideal for permeability
    - Rotatable bonds: <=5 is better (less flexible)
    """
    
    # Calculate lead score
    df_leads = df.copy()
    
    # Normalize descriptors to 0-1 range
    mw_score = 1 - np.abs(df_leads['molecular_weight'] - 450) / 200
    mw_score = np.clip(mw_score, 0, 1)
    
    logp_score = 1 - np.abs(df_leads['logp'] - 2.5) / 2
    logp_score = np.clip(logp_score, 0, 1)
    
    tpsa_score = 1 - np.abs(df_leads['tpsa'] - 90) / 50
    tpsa_score = np.clip(tpsa_score, 0, 1)
    
    rot_score = 1 - (df_leads['rotatable_bonds'] / 10)
    rot_score = np.clip(rot_score, 0, 1)
    
    # Composite score (weighted)
    df_leads['lead_score'] = (0.3 * mw_score + 
                               0.25 * logp_score + 
                               0.25 * tpsa_score + 
                               0.2 * rot_score)
    
    # Rank
    df_leads_ranked = df_leads.nlargest(n_leads, 'lead_score')
    
    return df_leads_ranked

# Get top leads
if len(lipinski_passed) > 0:
    top_leads = rank_lead_compounds(lipinski_passed, n_leads=10)
    
    print("\nTOP 10 LEAD COMPOUNDS (Lipinski-Compliant)\n")
    print("="*120)
    
    for rank, (idx, compound) in enumerate(top_leads.iterrows(), 1):
        print(f"\n#{rank} - {compound['compound_id']}")
        print(f"  Building Block:  {compound['building_block_id']}")
        print(f"  Lead Score:      {compound['lead_score']:.3f}")
        print(f"  MW:              {compound['molecular_weight']:.1f} Da")
        print(f"  LogP:            {compound['logp']:.2f}")
        print(f"  TPSA:            {compound['tpsa']:.1f} Ų")
        print(f"  H-Donors:        {compound['h_donors']}")
        print(f"  H-Acceptors:     {compound['h_acceptors']}")
        print(f"  Rotatable Bonds: {compound['rotatable_bonds']}")
        print(f"  SMILES:          {compound['product_smiles'][:70]}...")
else:
    print("No Lipinski-passing compounds found.")

In [ ]:
# Visualize top leads
if len(lipinski_passed) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Calculate lead scores for all compounds
    df_leads_all = lipinski_passed.copy()
    
    mw_score = 1 - np.abs(df_leads_all['molecular_weight'] - 450) / 200
    mw_score = np.clip(mw_score, 0, 1)
    
    logp_score = 1 - np.abs(df_leads_all['logp'] - 2.5) / 2
    logp_score = np.clip(logp_score, 0, 1)
    
    tpsa_score = 1 - np.abs(df_leads_all['tpsa'] - 90) / 50
    tpsa_score = np.clip(tpsa_score, 0, 1)
    
    rot_score = 1 - (df_leads_all['rotatable_bonds'] / 10)
    rot_score = np.clip(rot_score, 0, 1)
    
    df_leads_all['lead_score'] = (0.3 * mw_score + 
                                   0.25 * logp_score + 
                                   0.25 * tpsa_score + 
                                   0.2 * rot_score)
    
    # 1. Lead score distribution
    ax = axes[0, 0]
    ax.hist(df_leads_all['lead_score'], bins=15, color='steelblue', edgecolor='black', alpha=0.7)
    ax.axvline(df_leads_all['lead_score'].mean(), color='red', linestyle='--', 
               linewidth=2, label=f"Mean: {df_leads_all['lead_score'].mean():.3f}")
    ax.set_xlabel('Lead Score', fontsize=11, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
    ax.set_title('Distribution of Lead Scores', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # 2. MW vs LogP (top 10 highlighted)
    ax = axes[0, 1]
    ax.scatter(df_leads_all['molecular_weight'], df_leads_all['logp'], 
               c='lightgray', s=50, alpha=0.5, edgecolors='black', label='Other compounds')
    
    top_10 = df_leads_all.nlargest(10, 'lead_score')
    scatter = ax.scatter(top_10['molecular_weight'], top_10['logp'], 
                         c=top_10['lead_score'], cmap='RdYlGn', s=200, 
                         edgecolors='black', linewidth=2, vmin=0, vmax=1, label='Top 10 leads')
    
    # Add compound IDs to top leads
    for idx, row in top_10.iterrows():
        ax.annotate(row['compound_id'].split('_')[1], 
                   (row['molecular_weight'], row['logp']),
                   fontsize=8, ha='center', va='bottom')
    
    ax.set_xlabel('Molecular Weight (Da)', fontsize=11, fontweight='bold')
    ax.set_ylabel('LogP', fontsize=11, fontweight='bold')
    ax.set_title('Top 10 Leads (MW vs LogP)', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Lead Score', fontsize=10)
    
    # 3. TPSA vs Rotatable Bonds (top 10 highlighted)
    ax = axes[1, 0]
    ax.scatter(df_leads_all['tpsa'], df_leads_all['rotatable_bonds'], 
               c='lightgray', s=50, alpha=0.5, edgecolors='black', label='Other compounds')
    
    scatter = ax.scatter(top_10['tpsa'], top_10['rotatable_bonds'], 
                         c=top_10['lead_score'], cmap='RdYlGn', s=200, 
                         edgecolors='black', linewidth=2, vmin=0, vmax=1, label='Top 10 leads')
    
    # Add annotations
    for idx, row in top_10.iterrows():
        ax.annotate(row['compound_id'].split('_')[1], 
                   (row['tpsa'], row['rotatable_bonds']),
                   fontsize=8, ha='center', va='bottom')
    
    ax.set_xlabel('TPSA (Ų)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Rotatable Bonds', fontsize=11, fontweight='bold')
    ax.set_title('Top 10 Leads (Flexibility Profile)', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Lead Score', fontsize=10)
    
    # 4. Building block contribution to top leads
    ax = axes[1, 1]
    bb_counts_leads = top_10['building_block_id'].apply(lambda x: categorize_bb(x)).value_counts()
    colors_pie = plt.cm.Set3(np.linspace(0, 1, len(bb_counts_leads)))
    
    wedges, texts, autotexts = ax.pie(bb_counts_leads.values, 
                                        labels=bb_counts_leads.index,
                                        autopct='%1.0f%%',
                                        colors=colors_pie,
                                        textprops={'fontsize': 10, 'fontweight': 'bold'})
    ax.set_title('Building Block Types in Top 10 Leads', fontsize=12, fontweight='bold')
    
    plt.suptitle('Lead Compound Ranking & Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('lead_compound_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("✓ Saved: lead_compound_analysis.png")

## 13. Summary & Next Steps

In [ ]:
print("\n" + "="*100)
print("PHASE 2 SUMMARY: VIRTUAL LIBRARY EXPANSION & LIPINSKI FILTERING")
print("="*100)

print(f"""
[LIBRARY GENERATION METRICS]
├─ Total compounds generated: {len(library_df)}
├─ Unique SMILES: {library_df['product_smiles'].nunique()}
├─ Building blocks used: {len(BUILDING_BLOCKS)}
└─ Average products per building block: {len(library_df) / len(BUILDING_BLOCKS):.1f}

[LIPINSKI FILTERING RESULTS]
├─ Compounds passing Lipinski: {len(lipinski_passed)} ({100*len(lipinski_passed)/len(library_df):.1f}%)
├─ Compounds failing Lipinski: {len(lipinski_failed)} ({100*len(lipinski_failed)/len(library_df):.1f}%)
└─ Most common violation: MW out of range

[DESCRIPTOR PROFILES (Generated Library)]
├─ Molecular Weight:    {library_df['molecular_weight'].mean():.1f} ± {library_df['molecular_weight'].std():.1f} Da
├─ LogP (Lipophilicity): {library_df['logp'].mean():.2f} ± {library_df['logp'].std():.2f}
├─ TPSA (Polarity):     {library_df['tpsa'].mean():.1f} ± {library_df['tpsa'].std():.1f} Å²
├─ H-Donors:            {library_df['h_donors'].mean():.1f} ± {library_df['h_donors'].std():.1f}
├─ H-Acceptors:        {library_df['h_acceptors'].mean():.1f} ± {library_df['h_acceptors'].std():.1f}
└─ Rotatable Bonds:     {library_df['rotatable_bonds'].mean():.1f} ± {library_df['rotatable_bonds'].std():.1f}

[COMPARISON TO ORIGINAL TALOSIDE COMPOUNDS (MSc)]
├─ Original compounds:    {len(original_df)}
├─ MW difference:         {generated_df_copy['molecular_weight'].mean() - original_df['molecular_weight'].mean():+.1f} Da
├─ LogP difference:       {generated_df_copy['logp'].mean() - original_df['logp'].mean():+.2f}
└─ TPSA difference:       {generated_df_copy['tpsa'].mean() - original_df['tpsa'].mean():+.1f} Å²

[EXPORTED FILES]
├─ {all_compounds_path.name} ({len(library_df)} compounds)
├─ {lipinski_passed_path.name} ({len(lipinski_passed)} drug-like compounds)
└─ {lipinski_failed_path.name} ({len(lipinski_failed)} non-drug-like compounds)
""")

print("="*100)

## Appendix: Session Information & Reproducibility

In [ ]:
import sys
from rdkit import __version__ as rdkit_version

print("\nENVIRONMENT & REPRODUCIBILITY")
print("="*80)
print(f"Python version:     {sys.version.split(' ')[0]}")
print(f"RDKit version:      {rdkit_version}")
print(f"Pandas version:     {pd.__version__}")
print(f"NumPy version:      {np.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")
print(f"Seaborn version:    {sns.__version__}")
print("="*80)

print("\nAll outputs reproducible with git commit:")
print("  Repository: https://github.com/adamholohan6/taloside-screening-pipeline")
print(f"  Phase 2 notebook: Phase2_VirtualLibraryExpansion.ipynb")